Notebook for importing OSSIE YAML from external stages into Snowflake semantic views
*Co-authored with CoCo*

# 3. Snowflake: Import the Databricks Ossie from S3

This notebook reads the Ossie file that Databricks produced from S3 and creates a
semantic view from it. The measure added in Databricks (`TOTAL_QUANTITY`) comes across.

The import is a single built-in function -- Snowflake reads Ossie natively.
No file download or upload needed; both platforms share the same S3 bucket.

## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "EXT_SEMANTIC_INTEROP"
SEMANTIC_VIEW_NAME = 'SALES_SV'
STAGE_NAME = '{{STAGE_NAME}}'
OUTPUT_FILE = 'ossie_from_snowflake.yaml'
INPUT_FILE = 'ossie_from_databricks.yaml'

# Set to a different name if you don't want the return to overwrite your initial SV
TARGET_VIEW = 'SALES_SV'
# TARGET_VIEW = 'SALES_SV_2'


print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
%%sql -r dataframe_4
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

SET target_view = '{{TARGET_VIEW}}';

SELECT  CONCAT('Imported Semantic View will be titled {{DATABASE}}.{{SCHEMA}}.', $target_view);

## Step 2 - Read the Databricks Ossie from S3

The file is at `s3://<your-bucket>/ossie/ossie_from_databricks.yaml`,
accessible via the external stage.

In [ ]:
%%sql -r dataframe_2
LIST @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}};

## Step 3 - Choose the target view name & load the YAML

Default `SALES_SV_V2` leaves the original `SALES_SV` untouched.

In [ ]:
%%sql -r dataframe_3
SET yaml_content = (
  SELECT $1 FROM @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}/{{INPUT_FILE}}
  (FILE_FORMAT => '{{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT')
);

SET yaml_to_import = (SELECT REPLACE($yaml_content, 'SALES_SV_V2', '{{TARGET_VIEW}}')); -- Translate the semantic view name to prevent conflict (if desired)

SELECT $yaml_to_import;

## Step 4 - Import

In [ ]:
%%sql -r dataframe_7
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('{{DATABASE}}.{{SCHEMA}}', $yaml_to_import);

In [ ]:
%%sql -r dataframe_1
SHOW SEMANTIC METRICS IN {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}};

SELECT $5 as metric_name, $6 as data_type, $2 as database, $3 as schema, $4 as semantic_view_name, * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

## Step 5 - Verify

The round-tripped view returns the same numbers as Databricks: EAST 12/750/5, WEST 11/700/5.

In [ ]:
%%sql -r dataframe_8
SET target_fqn = '{{DATABASE}}.{{SCHEMA}}.' || $target_view;
SELECT * FROM SEMANTIC_VIEW(
  IDENTIFIER($target_fqn)
  DIMENSIONS region
  METRICS total_quantity, total_order_amount, order_count
) ORDER BY region;

# Automate

1. Create a Stored Procedure which reads Ossie files from S3 into Semantic Views
2. Build a task that checks for updated files on S3 and triggers the stored procedure.

**NOTE** This is **not** yet production ready, and if left on will result in a loop every time you update a Semantic View. It will trigger a load to Databricks, and that updated Metric View will trigger a load back to Snowflake. I will add a versioning/logging table in the future.

In [ ]:
%%sql -r import_proc_result
CREATE OR REPLACE PROCEDURE IMPORT_OSSIE_FROM_STAGE(
    P_DATABASE STRING,
    P_SCHEMA STRING,
    P_TARGET_SEMANTIC_VIEW_NAME STRING,
    P_STAGE_NAME STRING,
    P_INPUT_FILE_NAME STRING
)
RETURNS STRING
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET stage_path STRING := P_DATABASE || '.' || P_SCHEMA || '.' || P_STAGE_NAME || '/' || P_INPUT_FILE_NAME;
    LET target_schema STRING := P_DATABASE || '.' || P_SCHEMA;
    LET file_format STRING := P_DATABASE || '.' || P_SCHEMA || '.RAW_TEXT_FMT';

    -- Read the YAML from stage
    LET read_sql STRING := 'SELECT $1 FROM @' || :stage_path || ' (FILE_FORMAT => ''' || :file_format || ''')';
    LET res RESULTSET := (EXECUTE IMMEDIATE :read_sql);
    LET cur CURSOR FOR res;
    OPEN cur;
    LET yaml_content STRING;
    FETCH cur INTO :yaml_content;
    CLOSE cur;

    -- Replace embedded semantic view name with the target name
    LET yaml_to_import STRING := REPLACE(:yaml_content, 'SALES_SV_V2', P_TARGET_SEMANTIC_VIEW_NAME);

    -- Import into Snowflake as a semantic view
    CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML(:target_schema, :yaml_to_import);

    RETURN 'Imported OSSIE YAML from @' || :stage_path || ' as ' || P_DATABASE || '.' || P_SCHEMA || '.' || P_TARGET_SEMANTIC_VIEW_NAME;
END;

In [ ]:
-- CALL IMPORT_OSSIE_FROM_STAGE(
--     '{{DATABASE}}',
--     '{{SCHEMA}}',
--     '{{TARGET_VIEW}}',
--     '{{STAGE_NAME}}',
--     'ossie_from_databricks.yaml'
-- );

### Auto-Import Task
This task checks every minute if `ossie_from_databricks.yaml` has been updated on the external stage, and re-imports it if so.

In [ ]:
CREATE OR REPLACE TASK DEMOS.EXT_SEMANTIC_INTEROP.MONITOR_OSSIE_IMPORT
  WAREHOUSE = SI_DEMO_WH
  SCHEDULE = '1 MINUTE'
AS
BEGIN
    LET sv_created TIMESTAMP_LTZ;
    LET file_modified TIMESTAMP_LTZ;

    -- Get the semantic view's created_on timestamp (resets on each CREATE OR REPLACE)
    SHOW SEMANTIC VIEWS LIKE 'SALES_SV' IN SCHEMA DEMOS.EXT_SEMANTIC_INTEROP;
    SELECT "created_on" INTO :sv_created FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

    -- Get the file's last_modified from the directory table
    SELECT LAST_MODIFIED INTO :file_modified
      FROM DIRECTORY(@DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_S3_STAGE)
      WHERE RELATIVE_PATH = 'ossie_from_databricks.yaml';

    -- If the file is newer than the semantic view, re-import
    IF (:file_modified IS NULL) THEN
        RETURN 'File not found on stage';
    ELSEIF (:file_modified > :sv_created) THEN
        CALL IMPORT_OSSIE_FROM_STAGE(
            'DEMOS',
            'EXT_SEMANTIC_INTEROP',
            'SALES_SV',
            'OSSIE_S3_STAGE',
            'ossie_from_databricks.yaml'
        );
        -- Refresh directory table metadata
        ALTER STAGE DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_S3_STAGE REFRESH;
        RETURN 'Imported updated ossie_from_databricks.yaml';
    END IF;

    RETURN 'No update detected';
END;

In [ ]:
%%sql -r resume_task_result
ALTER TASK DEMOS.EXT_SEMANTIC_INTEROP.MONITOR_OSSIE_IMPORT RESUME;

In [ ]:
ALTER TASK DEMOS.EXT_SEMANTIC_INTEROP.MONITOR_OSSIE_IMPORT SUSPEND;

In [ ]:
%%sql -r dataframe_15
-- DROP SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.SALES_SV;

In [ ]:
SHOW SEMANTIC VIEWS IN {{DATABASE}}.{{SCHEMA}};

In [ ]:
SHOW SEMANTIC METRICS IN {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}};

SELECT $5 as metric_name, $6 as data_type, $2 as database, $3 as schema, $4 as semantic_view_name, * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));